# Getting Started with GeoDeep

[GeoDeep](https://github.com/uav4geo/GeoDeep) is an ultra-lightweight library for object detection on georeferenced rasters. Its key design goal is **minimal dependencies**: it only requires `rasterio` and `onnxruntime` — no PyTorch, no CUDA, no heavy ML framework.

**Why use GeoDeep?**
- You need to deploy object detection on edge hardware or in environments without a full deep learning stack
- You want to get results immediately with pre-built models (cars, buildings, trees, aircraft)
- You are comfortable working with ONNX models for custom architectures

Output is always a GeoJSON file with bounding-box features and the original CRS preserved.

## References
- GitHub: https://github.com/uav4geo/GeoDeep

## 1. Verify Installation

In [ ]:
import geodeep
print(f"GeoDeep version: {geodeep.__version__}")

# List available pre-built models
models = geodeep.list_models()
print("\nAvailable pre-built models:")
for m in models:
    print(f"  - {m}")

## 2. Download Sample GeoTIFF

In [ ]:
import os
import urllib.request

# Sample aerial orthophoto from OpenAerialMap
url = "https://github.com/uav4geo/GeoDeep/releases/download/v0.1.0/sample.tif"
image_path = "sample.tif"

if not os.path.exists(image_path):
    print("Downloading sample image...")
    urllib.request.urlretrieve(url, image_path)
    print("Done.")

import rasterio
from rasterio.plot import show
import matplotlib.pyplot as plt

with rasterio.open(image_path) as src:
    fig, ax = plt.subplots(figsize=(8, 8))
    show(src, ax=ax)
    ax.set_title("Sample GeoTIFF")
    plt.tight_layout()
    plt.show()
    print(f"CRS: {src.crs}")
    print(f"Size: {src.width} x {src.height} px")
    print(f"Resolution: {src.res} (units: {src.crs.linear_units if src.crs else 'unknown'})")

## 3. Run Detection with a Pre-Built Model

In [ ]:
output_geojson = "detections.geojson"

# Use the 'cars' model — downloads ONNX weights automatically on first use
geodeep.detect(
    input=image_path,
    output=output_geojson,
    model="cars",
    confidence=0.5,
)

print(f"Detection complete. Results saved to {output_geojson}")

## 4. Inspect the GeoJSON Output

In [ ]:
import json

with open(output_geojson) as f:
    geojson = json.load(f)

features = geojson["features"]
print(f"Detections: {len(features)}")

if features:
    sample = features[0]
    print("\nSample feature:")
    print(f"  Geometry type: {sample['geometry']['type']}")
    print(f"  Properties: {sample['properties']}")
    print(f"  Coordinates (first ring): {sample['geometry']['coordinates'][0][:2]}...")

## 5. Visualize Detections on the Raster

In [ ]:
import geopandas as gpd
import numpy as np

gdf = gpd.read_file(output_geojson)
print(f"Detection CRS: {gdf.crs}")

with rasterio.open(image_path) as src:
    img_array = src.read([1, 2, 3])
    img_array = np.moveaxis(img_array, 0, -1)
    extent = [src.bounds.left, src.bounds.right, src.bounds.bottom, src.bounds.top]

fig, ax = plt.subplots(figsize=(10, 10))
ax.imshow(img_array, extent=extent)
gdf.boundary.plot(ax=ax, color="red", linewidth=1.5)
ax.set_title(f"GeoDeep 'cars' model — {len(gdf)} detections")
plt.tight_layout()
plt.show()

## 6. Available Models Summary

| Model | Target objects | Notes |
|---|---|---|
| `cars` | Vehicles / cars | Trained on aerial imagery |
| `buildings` | Building footprints | Rectangular bounding boxes |
| `trees` | Individual tree crowns | Circular crown detection |
| `aircraft` | Aircraft on runways | Airport/airfield scenes |

All models output georeferenced GeoJSON with confidence scores.
See `geodeep.list_models()` for the current list.